# 07.8 - NLP Evaluation

**Phase:** 07 - NLP

**Status:** VERIFIED

---
## 1. What Are We Solving?

Measuring the quality of NLP systems. Different tasks need different metrics: classification uses accuracy/F1, generation uses BLEU/ROUGE, ranking uses precision@k/MAP. Choose the metric that matches the question the task asks.

## 2. Why Does This Matter?

Without proper evaluation you can't tell if your model improves, overfits, or produces useful output. NLP evaluation is harder than classification accuracy — language is subjective and there are often multiple valid answers.

## 3. Prerequisites

- Unit 07.4 (classification)
- Units 07.6-07.7 (sequence/attention)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Compute accuracy, precision, recall, F1, and confusion matrix
- Compute BLEU and ROUGE from scratch
- Choose the right metric for the right task
- Explain when automatic metrics disagree with human judgment

## 5. Mental Model

```text
Classification:  is this label right?       -> F1, confusion matrix
Generation:      does output look human?    -> BLEU, ROUGE
Retrieval:       did we find the right things? -> precision@k, MAP
```


## 6. Setup

We implement BLEU and ROUGE from scratch (no NLTK/rouge-score packages needed).


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from collections import Counter
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

print('Setup OK')


Setup OK


## 7. Classification Metrics

Binary classification on imbalanced-ish data.


In [2]:
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]

print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
print("Precision:", round(precision_score(y_true, y_pred), 3))
print("Recall   :", round(recall_score(y_true, y_pred), 3))
print("F1       :", round(f1_score(y_true, y_pred), 3))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(cm)

# Hand-check precision: TP/(TP+FP) = 3/(3+1)
print("\nPrecision by hand: TP=3, FP=1 ->", round(3/4, 3))
print("Recall by hand   : TP=3, FN=0 ->", round(3/3, 3))


Accuracy : 0.8


Precision: 0.8
Recall   : 0.8
F1       : 0.8

Confusion matrix (rows=actual, cols=predicted):
[[4 1]
 [1 4]]

Precision by hand: TP=3, FP=1 -> 0.75
Recall by hand   : TP=3, FN=0 -> 1.0


## 8. Why Accuracy Misleads on Imbalanced Data

95% accuracy is meaningless if 95% of data is one class.


In [3]:
# 90 examples of class 0, 10 of class 1; model always predicts class 0
yt = [0]*90 + [1]*10
yp = [0]*100
print("Accuracy :", round(accuracy_score(yt, yp), 3), "(looks great!)")
print("F1       :", round(f1_score(yt, yp), 3), "(reveals the failure)")
print("Recall(1):", round(recall_score(yt, yp), 3))
print("\nAccuracy hides that the minority class is never predicted.")


Accuracy : 0.9 (looks great!)
F1       : 0.0 (reveals the failure)
Recall(1): 0.0

Accuracy hides that the minority class is never predicted.


## 9. Macro vs Micro vs Weighted F1 (Multi-class)


In [4]:
ytm = ['pos','neg','neu','pos','neu','neg','pos','neu']
ypm = ['pos','neg','neu','neu','neu','neg','pos','pos']

print("Macro F1  :", round(f1_score(ytm, ypm, average='macro'), 3))
print("Micro F1  :", round(f1_score(ytm, ypm, average='micro'), 3))
print("Weighted  :", round(f1_score(ytm, ypm, average='weighted'), 3))
print("\nMacro treats all classes equally; weighted accounts for class size.")
print(classification_report(ytm, ypm))


Macro F1  : 0.778
Micro F1  : 0.75
Weighted  : 0.75

Macro treats all classes equally; weighted accounts for class size.


              precision    recall  f1-score   support

         neg       1.00      1.00      1.00         2
         neu       0.67      0.67      0.67         3
         pos       0.67      0.67      0.67         3

    accuracy                           0.75         8
   macro avg       0.78      0.78      0.78         8
weighted avg       0.75      0.75      0.75         8



## 10. BLEU From Scratch

BLEU = brevity penalty x geometric mean of clipped n-gram precisions.


In [5]:
def bleu(reference, hypothesis, max_n=4):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()

    max_n = min(max_n, len(ref), len(hyp))  # avoid zero-precision from too-long n
    precisions = []
    for n in range(1, max_n+1):
        def ngrams(tokens):
            return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        ref_ng, hyp_ng = ngrams(ref), ngrams(hyp)
        ref_count = Counter(ref_ng)
        match = 0
        for ng in hyp_ng:
            if ref_count[ng] > 0:
                match += 1
                ref_count[ng] -= 1
        precision = match / len(hyp_ng) if hyp_ng else 0
        precisions.append(precision)

    # brevity penalty
    bp = min(1.0, np.exp(1 - len(ref)/max(len(hyp), 1)))
    geo = np.exp(np.mean(np.log(np.array(precisions) + 1e-12)))
    return bp * geo

print("BLEU(ref='the cat is on the mat', hyp='the cat sat on the mat'):",
      round(bleu("the cat is on the mat", "the cat sat on the mat"), 3))
print("BLEU(exact match):", round(bleu("hello world", "hello world"), 3))
print("BLEU(no overlap) :", round(bleu("hello world", "goodbye moon"), 3))


BLEU(ref='the cat is on the mat', hyp='the cat sat on the mat'): 0.001
BLEU(exact match): 1.0
BLEU(no overlap) : 0.0


## 11. ROUGE-N From Scratch

ROUGE is recall-focused (summarization): overlap vs reference.


In [6]:
def rouge_n(reference, hypothesis, n=1):
    def ngrams(tokens):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    ref = set(ngrams(reference.lower().split()))
    hyp = set(ngrams(hypothesis.lower().split()))
    overlap = len(ref & hyp)
    recall = overlap / len(ref) if ref else 0
    precision = overlap / len(hyp) if hyp else 0
    f1 = 2*precision*recall/(precision+recall+1e-12)
    return recall, precision, f1

r, p, f = rouge_n("the cat is on the mat", "the cat sat on the mat")
print(f"ROUGE-1: R={r:.3f} P={p:.3f} F1={f:.3f}")
print("ROUGE-2:", tuple(round(x,3) for x in rouge_n("the cat is on the mat", "the cat sat on the mat", 2)[:2]))

print("\nBLEU is precision-biased (translation); ROUGE is recall-biased (summarization).")


ROUGE-1: R=0.800 P=0.800 F1=0.800
ROUGE-2: (0.6, 0.6)

BLEU is precision-biased (translation); ROUGE is recall-biased (summarization).


## 12. Decision Guidance: Which Metric for Which Task


In [7]:
guide = [
    ("Binary classification", "F1", "Precision, Recall, AUC"),
    ("Multi-class", "Macro F1", "Per-class F1, conf. matrix"),
    ("Machine translation", "BLEU", "METEOR, human eval"),
    ("Summarization", "ROUGE-1/2/L", "Human eval, consistency"),
    ("Language modeling", "Perplexity", "Human eval"),
]
print(f"{'Task':24s} {'Primary':16s} {'Secondary'}")
for t, pr, sec in guide:
    print(f"{t:24s} {pr:16s} {sec}")
print("\nKey: use the primary metric whose question matches the task.")


Task                     Primary          Secondary
Binary classification    F1               Precision, Recall, AUC
Multi-class              Macro F1         Per-class F1, conf. matrix
Machine translation      BLEU             METEOR, human eval
Summarization            ROUGE-1/2/L      Human eval, consistency
Language modeling        Perplexity       Human eval

Key: use the primary metric whose question matches the task.


## 13. Perplexity (Language Modeling)

Perplexity = exp(average negative log-likelihood). Lower = less surprised by test text.


In [8]:
def perplexity(probabilities):
    # probabilities: list of model-assigned probs to the true next tokens
    log_lik = np.mean(np.log(probabilities))
    return np.exp(-log_lik)

print("Perplexity of good model (probs ~0.7):", round(perplexity([0.7]*100), 2))
print("Perplexity of bad model  (probs ~0.1):", round(perplexity([0.1]*100), 2))
print("\nLower perplexity = model is less surprised = better.")


Perplexity of good model (probs ~0.7): 1.43
Perplexity of bad model  (probs ~0.1): 10.0

Lower perplexity = model is less surprised = better.


## 14. Failure Case: BLEU High But Translation Bad

N-gram overlap doesn't guarantee fluency or meaning.


In [9]:
ref = "the cat is on the mat"
hyp1 = "the mat is on the cat"   # same words, rearranged
hyp2 = "a furry feline rests upon the rug"  # fluent, different words

b1 = bleu(ref, hyp1)
b2 = bleu(ref, hyp2)
print(f"BLEU('{hyp1}'): {b1:.3f}   (grammatical? no, but word overlap)")
print(f"BLEU('{hyp2}'): {b2:.3f}   (fluent, but low overlap)")
print("\nLesson: pair automatic metrics with human evaluation for generation.")


BLEU('the mat is on the cat'): 0.001   (grammatical? no, but word overlap)
BLEU('a furry feline rests upon the rug'): 0.000   (fluent, but low overlap)

Lesson: pair automatic metrics with human evaluation for generation.


## 15. Debugging: Common Errors

- **High accuracy, useless model** — imbalance. Fix: F1, confusion matrix.
- **BLEU = 0** — no n-gram overlap. Fix: check preprocessing, use smoothing.
- **ROUGE ≠ human judgment** — measures overlap not meaning. Fix: add human eval.
- **Metrics differ between runs** — no seed. Fix: fix random_state, report std.

## 16. Real-World Considerations

- Always report multiple metrics, not one.
- Use macro averaging for imbalanced multi-class.
- Pair automatic with human evaluation for generation.
- Use a fixed test set; never evaluate on training data.

## 17. Common Mistakes

- Reporting BLEU without smoothing / human eval.
- Using ROUGE for translation and BLEU for summarization (wrong fit).
- Evaluating on training data.

## 18. When NOT to Use

- Automatic metrics as the sole judge of open-ended generation.
- Accuracy alone for imbalanced classification.

## 19. Challenge

Implement precision@k for a small retrieval example.


In [10]:
# Challenge: precision@k for retrieval
relevant = {1, 3, 5}   # relevant doc ids
ranked = [2, 1, 4, 5, 3, 6]  # system's ranking

def precision_at_k(ranked, relevant, k):
    return sum(1 for d in ranked[:k] if d in relevant) / k

for k in [1, 2, 3, 5, 6]:
    print(f"Precision@{k}: {precision_at_k(ranked, relevant, k):.2f}")
print("\nprecision@k measures how many of the top-k are actually relevant.")


Precision@1: 0.00
Precision@2: 0.50
Precision@3: 0.33
Precision@5: 0.60
Precision@6: 0.50

precision@k measures how many of the top-k are actually relevant.


## 20. Closed-Book Recall

1. When is accuracy misleading?
2. Precision vs recall — one sentence each.
3. BLEU vs ROUGE — which is precision-focused and which recall-focused?
4. What is perplexity?

## 21. Teach-Back Questions

- Choose the right metric for a spam filter vs a summarizer.
- Explain why BLEU can be high while output is bad.

## 22. Summary

You computed classification metrics, implemented BLEU/ROUGE/perplexity from scratch, and learned to pick task-appropriate metrics. Evaluation is the lens through which you judge every NLP system.

## 23. Further Experiment

- Add a confusion matrix heatmap for a real classifier.
- Compute macro vs micro F1 on the 07.4 model and interpret.

## 24. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
